# 15 Full Fusion

Fuses available upstream `predictions_v1` rows per target without crossing event boundaries.

Default source runs:

- `context_catboost_mlb_2024_2026_v2` from notebook 05b
- `sequence_tcn_mlb_2024_2026_v2`, with `sequence_structured_mlb_2024_2026_v2` as fallback
- `video_lightweight_cv2_mlb_2024_2026_v2`
- `video_frozen_encoder_mlb_2024_2026_v2`
- `video_raw_finetune_mlb_2024_2026_v2`
- `player_season_aggregate_mlb_2024_2026_v2`
- `vlm_mechanics_mlb_2024_2026_v2`

Created outputs:

- `predictions/fusion_mlb_2024_2026_v2/predictions_v1.parquet`
- `predictions/fusion_mlb_2024_2026_v2/metrics_v1.json`
- `predictions/fusion_mlb_2024_2026_v2/fusion_input_audit_v1.parquet`

Next: run `09_report_builder.ipynb` and `22_research_outputs.ipynb`. For pre-fusion visual/mechanics-only evaluation, run `25_method_evaluation.ipynb` before this notebook.

In [ ]:
from pathlib import Path
import os
import sys
import importlib.util


def _mount_drive_if_needed() -> None:
    try:
        from google.colab import drive  # type: ignore
    except ModuleNotFoundError:
        print('Google Colab ではないため Drive mount を skip します。')
        return

    mountpoint = Path('/content/drive')
    mountpoint.mkdir(parents=True, exist_ok=True)
    if (mountpoint / 'MyDrive').exists():
        print('Drive already mounted at /content/drive')
        return
    try:
        drive.mount(str(mountpoint))
    except ValueError as exc:
        message = str(exc)
        if 'Mountpoint must not already contain files' in message or 'already mounted' in message:
            print(f'Drive mount warning: {message}')
            if not (mountpoint / 'MyDrive').exists():
                drive.mount(str(mountpoint), force_remount=True)
        else:
            raise
    except Exception as exc:
        print(f'Colab Drive mount skipped or failed: {exc}')


def _is_repo_dir(path: Path) -> bool:
    return (
        (path / 'src' / 'sport_pipeline' / '__init__.py').exists()
        and (path / 'configs').exists()
        and (path / 'notebooks').exists()
    )


def _resolve_repo_dir() -> Path:
    fixed = Path('/content/drive/MyDrive/codex/batting_codex_handoff')
    candidates: list[Path] = []
    env_root = os.environ.get('BATTING_CODE_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend(
        [
            fixed,
            Path.cwd(),
        ]
    )
    candidates.extend(Path.cwd().parents)

    checked: list[str] = []
    for candidate in candidates:
        candidate = candidate.expanduser().resolve()
        if str(candidate) in checked:
            continue
        checked.append(str(candidate))
        if _is_repo_dir(candidate):
            if candidate != fixed:
                print('WARNING: fixed REPO_DIR ではない場所の repo を使います。')
                print('  fixed:', fixed)
                print('  using:', candidate)
                print('  次回は repo フォルダを fixed path に置くか、BATTING_CODE_ROOT を明示してください。')
            return candidate

    raise FileNotFoundError(
        'sport_pipeline package が見つかりません。Drive には notebook 単体ではなく repo フォルダごと置く必要があります。\n'
        '推奨配置: /content/drive/MyDrive/codex/batting_codex_handoff\n'
        '別の場所に置いた場合は、この notebook の最初の cell より前に次を実行してください。\n'
        '  %env BATTING_CODE_ROOT=/content/drive/MyDrive/batting_codex_handoff\n'
        '確認した候補:\n- ' + '\n- '.join(checked)
    )


_mount_drive_if_needed()

REPO_DIR = _resolve_repo_dir()
BASE_DIR = Path('/content/drive/MyDrive/baseball_vision')
CACHE_DIR = Path('/content/cache/baseball_vision')
RUN_PROFILE_NAME = os.environ.get('BASEBALL_VISION_RUN_PROFILE', 'mlb_2024_2026_real_colab_v2.json')
RUN_PROFILE_PATH = REPO_DIR / 'configs' / 'runs' / RUN_PROFILE_NAME

BASE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(REPO_DIR)

src_dir = REPO_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if importlib.util.find_spec('sport_pipeline') is None:
    raise ModuleNotFoundError(
        'sport_pipeline を import できません。repo 配置と src/sport_pipeline/__init__.py を確認してください。\n'
        f'REPO_DIR={REPO_DIR}\n'
        f'src_dir={src_dir}\n'
        f'src_dir_exists={src_dir.exists()}'
    )

print('REPO_DIR =', REPO_DIR)
print('BASE_DIR =', BASE_DIR)
print('CACHE_DIR =', CACHE_DIR)
print('RUN_PROFILE_PATH =', RUN_PROFILE_PATH)
print('src_dir =', src_dir)

from sport_pipeline.pipeline.run_profile import load_run_profile, resolve_statcast_date_range, run_id, stage_settings, threshold

RUN_PROFILE = load_run_profile(RUN_PROFILE_PATH)
START_DATE, END_DATE = resolve_statcast_date_range(RUN_PROFILE)
FUSION_RUN_ID = run_id(RUN_PROFILE, 'fusion_run_id')

print('STATCAST_DATE_RANGE =', START_DATE, 'to', END_DATE)
print('FUSION_RUN_ID =', FUSION_RUN_ID)

FUSION_SETTINGS = stage_settings(RUN_PROFILE, 'fusion')
SOURCE_RUNS = list(FUSION_SETTINGS.get('source_runs', []))
if not SOURCE_RUNS:
    SOURCE_RUNS = [run_id(RUN_PROFILE, 'context_run_id'), run_id(RUN_PROFILE, 'sequence_tcn_run_id'), run_id(RUN_PROFILE, 'video_run_id')]
LEARN_WEIGHTS_FROM_VALIDATION = bool(FUSION_SETTINGS.get('learn_weights_from_validation', False))
print('SOURCE_RUNS =', SOURCE_RUNS)
print('LEARN_WEIGHTS_FROM_VALIDATION =', LEARN_WEIGHTS_FROM_VALIDATION)


## Source Check

Missing source runs are skipped, but a useful fusion run needs at least one upstream predictions file.

In [ ]:
for source_run_id in SOURCE_RUNS:
    candidates = [BASE_DIR / 'predictions' / source_run_id / f'predictions_v1{suffix}' for suffix in ('.parquet', '.jsonl', '.json', '.csv')]
    existing = next((path for path in candidates if path.exists()), candidates[0])
    print(source_run_id, '->', existing, 'exists=', existing.exists())

## Run Fusion

In [ ]:
from sport_pipeline.models.fusion.run_fusion import run_full_fusion

outputs = run_full_fusion(
    BASE_DIR,
    fusion_run_id=FUSION_RUN_ID,
    source_runs=SOURCE_RUNS,
    learn_weights_from_validation=LEARN_WEIGHTS_FROM_VALIDATION,
)
for name, path in outputs.items():
    print(name, '->', path, 'exists=', path.exists())

## Output Check And Report Command

In [ ]:
from sport_pipeline.artifact_check import check_artifacts

expected = [
    f'predictions/{FUSION_RUN_ID}/predictions_v1.parquet',
    f'predictions/{FUSION_RUN_ID}/metrics_v1.json',
    f'predictions/{FUSION_RUN_ID}/fusion_input_audit_v1.parquet',
    f'reports/preflight/full_fusion_{FUSION_RUN_ID}.json',
]
print(check_artifacts(BASE_DIR, expected))
print('REPORT COMMAND:')
print(f'python -m sport_pipeline.reports.build_static --base-dir {BASE_DIR} --run-id {FUSION_RUN_ID} --predictions {BASE_DIR}/predictions/{FUSION_RUN_ID}/predictions_v1.parquet --metrics {BASE_DIR}/predictions/{FUSION_RUN_ID}/metrics_v1.json --fusion-audit {BASE_DIR}/predictions/{FUSION_RUN_ID}/fusion_input_audit_v1.parquet')